# MedicalNLP-RAG — Phase 5
## One-Time Locked Internal Test Evaluation

This notebook performs the **first and only label-based evaluation** of the 860-question
`internal_test_LOCKED` split after all development choices have been frozen.

### Frozen pipeline

`BM25 + MedCPT → RRF → MedCPT Cross-Encoder → Top-5 documents → Qwen3-4B-Instruct → Phase 4C v2 grounding → task-specific Ridge/isotonic confidence → frozen Phase 4E selective policies`

### Non-negotiable test protocol

**Stage A — label-blind inference**
1. read only `id`, `type`, and `body` from the locked split;
2. run the frozen retrieval/reranking pipeline;
3. generate one frozen `document_top5` answer per question;
4. calculate frozen Phase 4C v2 grounding features;
5. apply the already-fitted task-specific confidence models;
6. rank within task and freeze Policy A / Policy B answer–abstain masks;
7. hash and save all label-blind outputs.

**Stage B — one-time gold scoring**
1. verify that Stage A is complete and unchanged;
2. only then reopen the locked split with gold answers/documents;
3. compute final retrieval, answer-quality, grounding, calibration, and selective-answering results;
4. never refit, retune, re-rank, or change coverage from locked-test labels.

The notebook is resumable. Existing completed Stage A outputs are reused rather than recomputed.
A completed Phase 5 marker prevents accidental repeated test evaluation.

## 0. Configuration — keep both gates `False` initially

Run the notebook through the frozen-artifact verification cells first.

When ready for the one-time evaluation:

1. change `UNLOCK_STAGE_A` to `True` and set the confirmation phrase;
2. run Stage A to completion;
3. inspect only the Stage A integrity summary — **do not alter the method**;
4. then change `OPEN_GOLD_STAGE_B` to `True` and set its confirmation phrase;
5. run Stage B once.

In [ ]:
PROJECT_NAME = "MedicalNLP_RAG"
SEED = 42

# ---------- ONE-TIME TEST GATES ----------
UNLOCK_STAGE_A = True
STAGE_A_CONFIRMATION = "RUN_FROZEN_LOCKED_TEST_INFERENCE"

OPEN_GOLD_STAGE_B = True
STAGE_B_CONFIRMATION = "SCORE_FROZEN_LOCKED_TEST_ONCE"

ALLOW_RERUN_AFTER_COMPLETION = False

# ---------- Frozen retrieval ----------
BM25_K1 = 1.2
BM25_B = 0.75
RRF_K0 = 60
RETRIEVAL_TOP_K = 100

QUERY_MODEL = "ncbi/MedCPT-Query-Encoder"
QUERY_MAX_LENGTH = 64
DENSE_QUERY_BATCH_SIZE = 128

RERANK_MODEL = "ncbi/MedCPT-Cross-Encoder"
RERANK_MAX_LENGTH = 512
RERANK_BATCH_SIZE = 32

# ---------- Frozen evidence / generation ----------
DOCUMENT_TOP_K = 5
LLM_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
USE_4BIT = True
LLM_BATCH_SIZE = 2
MAX_EVIDENCE_TOKENS = 6000
MAX_INPUT_TOKENS = 8192
MAX_NEW_TOKENS = 320

# ---------- Frozen grounding ----------
GROUNDING_MEDCPT_MODEL = "ncbi/MedCPT-Cross-Encoder"
GROUNDING_MEDCPT_MAX_LENGTH = 512
GROUNDING_MEDCPT_BATCH_SIZE = 48
GROUNDING_TOP_M = 3

NLI_MODEL = "pritamdeka/PubMedBERT-MNLI-MedNLI"
NLI_MAX_LENGTH = 512
NLI_BATCH_SIZE = 48

EVIDENCE_WINDOW_SENTENCES = 2
EVIDENCE_WINDOW_STRIDE = 1
MAX_WINDOWS_PER_DOCUMENT = 12
MIN_SENTENCE_CHARS = 20

# ---------- Frozen Phase 4E policies ----------
TASKS = ["yesno", "factoid", "list", "summary"]
RANKING_SCORE = "task_ridge_raw"
CALIBRATED_CONFIDENCE = "task_ridge_isotonic"

POLICY_A = {
    "name": "conservative_summary80",
    "task_coverages": {
        "summary": 0.80,
        "factoid": 1.00,
        "yesno": 1.00,
        "list": 1.00,
    },
}
POLICY_B = {
    "name": "expanded_summary_factoid80",
    "task_coverages": {
        "summary": 0.80,
        "factoid": 0.80,
        "yesno": 1.00,
        "list": 1.00,
    },
}

BOOTSTRAP_REPS = 5000

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Dependency check

In [ ]:
import sys, subprocess, importlib, importlib.util
from packaging.version import Version

def package_version(name):
    try:
        return getattr(importlib.import_module(name), "__version__", None)
    except Exception:
        return None

need = []
tv = package_version("transformers")
if tv is None or Version(tv) < Version("4.51.0"):
    need.append("transformers>=4.51.0")
if importlib.util.find_spec("accelerate") is None:
    need.append("accelerate>=1.0.0")
if importlib.util.find_spec("bitsandbytes") is None:
    need.append("bitsandbytes>=0.45.0")
if importlib.util.find_spec("rouge_score") is None:
    need.append("rouge-score>=0.1.2")

if need:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])

print("Dependency check complete.")

## 3. Imports, deterministic seed, paths, and helpers

In [ ]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import gc, hashlib, json, math, os, platform, random, re, unicodedata, warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import sparse
from scipy.stats import spearmanr

import torch
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from sklearn.metrics import f1_score, mean_absolute_error, mean_squared_error, log_loss
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path(f"/content/drive/MyDrive/{PROJECT_NAME}")
SPLIT_DIR = ROOT / "data" / "bioasq14_splits"
PUBMED_CACHE_DIR = ROOT / "data" / "pubmed_cache"
INDEX_DIR = ROOT / "models" / "retrieval_indexes"

PHASE3C_DIR = ROOT / "results" / "phase3C_reranking"
PHASE4B_DIR = ROOT / "results" / "phase4B_evidence_filtering"
PHASE4C_DIR = ROOT / "results" / "phase4C_grounding_v2"
PHASE4D_DIR = ROOT / "results" / "phase4D_v2_task_specific_confidence"
PHASE4E_DIR = ROOT / "results" / "phase4E_task_specific_selective_answering"

PHASE5_DIR = ROOT / "results" / "phase5_one_time_locked_internal_test"
STAGEA_DIR = PHASE5_DIR / "stageA_label_blind"
STAGEB_DIR = PHASE5_DIR / "stageB_final_scoring"
TABLE_DIR = PHASE5_DIR / "tables"
FIG_DIR = PHASE5_DIR / "figures"
LOG_DIR = PHASE5_DIR / "logs"
CACHE_DIR = PHASE5_DIR / "cache"

for p in [PHASE5_DIR, STAGEA_DIR, STAGEB_DIR, TABLE_DIR, FIG_DIR, LOG_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def write_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def append_jsonl(rec, path):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def find_locked_split():
    candidates = [
        SPLIT_DIR / "internal_test_LOCKED.json",
        SPLIT_DIR / "internal_test_locked.json",
        SPLIT_DIR / "internal_test.json",
    ]
    existing = [p for p in candidates if p.exists()]
    if len(existing) != 1:
        raise FileNotFoundError(
            "Expected exactly one locked-test split file among: "
            + ", ".join(str(x) for x in candidates)
            + f". Found: {[str(x) for x in existing]}"
        )
    return existing[0]

LOCKED_TEST_JSON = find_locked_split()

CORPUS_CSV = PUBMED_CACHE_DIR / "pubmed_closed_corpus.csv"
CORPUS_INDEX_CSV = INDEX_DIR / "bioasq14_closed_corpus_index.csv"
BM25_MATRIX_PATH = INDEX_DIR / "bioasq14_closed_bm25_matrix.npz"
BM25_VECTORIZER_PATH = INDEX_DIR / "bioasq14_closed_bm25_vectorizer.joblib"
ARTICLE_EMBED_PATH = INDEX_DIR / "medcpt_article_embeddings_f16.npy"
ARTICLE_PMIDS_PATH = INDEX_DIR / "medcpt_article_pmids.json"

FROZEN_RETRIEVAL_JSON = PHASE3C_DIR / "frozen_retrieval_pipeline.json"
FROZEN_EVIDENCE_JSON = PHASE4B_DIR / "frozen_evidence_strategy.json"
FROZEN_GROUNDING_JSON = PHASE4C_DIR / "frozen_grounding_method_v2.json"
FROZEN_CONFIDENCE_JSON = PHASE4D_DIR / "frozen_confidence_method_v2.json"
FROZEN_POLICY_JSON = PHASE4E_DIR / "frozen_selective_answering_policy.json"

PHASE4D_MODEL_DIR = PHASE4D_DIR / "models"

RUN_MARKER = PHASE5_DIR / "one_time_evaluation_marker.json"
STAGEA_MANIFEST = STAGEA_DIR / "stageA_freeze_manifest.json"
FINAL_REPORT = PHASE5_DIR / "phase5_final_locked_test_report.json"

print("Project root:", ROOT)
print("Locked split:", LOCKED_TEST_JSON)
print("Phase 5 output:", PHASE5_DIR)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

## 4. Verify every frozen upstream artifact before touching the test split

This cell must pass with the development-stage freeze files unchanged.  
It intentionally verifies that the internal test was **not** previously evaluated.

In [ ]:
required = [
    LOCKED_TEST_JSON,
    CORPUS_CSV,
    CORPUS_INDEX_CSV,
    BM25_MATRIX_PATH,
    BM25_VECTORIZER_PATH,
    ARTICLE_EMBED_PATH,
    ARTICLE_PMIDS_PATH,
    FROZEN_RETRIEVAL_JSON,
    FROZEN_EVIDENCE_JSON,
    FROZEN_GROUNDING_JSON,
    FROZEN_CONFIDENCE_JSON,
    FROZEN_POLICY_JSON,
]
for p in required:
    assert p.exists(), f"Missing required artifact: {p}"

frozen_retrieval = load_json(FROZEN_RETRIEVAL_JSON)
frozen_evidence = load_json(FROZEN_EVIDENCE_JSON)
frozen_grounding = load_json(FROZEN_GROUNDING_JSON)
frozen_confidence = load_json(FROZEN_CONFIDENCE_JSON)
frozen_policy = load_json(FROZEN_POLICY_JSON)

assert frozen_retrieval["frozen_pipeline"] == "RRF+MedCPT_CrossEncoder"
assert frozen_evidence["frozen_evidence_strategy"] == "document_top5"
assert frozen_grounding["frozen_evidence_strategy"] == "document_top5"
assert frozen_grounding["evidence_selection"]["fixed_top_m"] == 3
assert frozen_grounding["nli"]["entailment_threshold_selected"] is False

assert frozen_confidence["internal_test_evaluated"] is False
assert frozen_confidence["question_type_used_as_feature"] is False
assert frozen_confidence["abstention_threshold_selected"] is False
assert frozen_policy["internal_test_evaluated"] is False
assert frozen_policy["coverage_selection"]["best_coverage_selected_post_hoc"] is False

assert frozen_policy["ranking_signal"] == RANKING_SCORE
assert frozen_policy["reported_confidence"] == CALIBRATED_CONFIDENCE
assert frozen_policy["primary_policy"]["name"] == POLICY_A["name"]
assert frozen_policy["secondary_policy"]["name"] == POLICY_B["name"]
assert frozen_policy["primary_policy"]["task_coverages"] == POLICY_A["task_coverages"]
assert frozen_policy["secondary_policy"]["task_coverages"] == POLICY_B["task_coverages"]

expected_eligibility = {
    "yesno": "INELIGIBLE",
    "factoid": "SECONDARY_ELIGIBLE",
    "list": "INELIGIBLE",
    "summary": "PRIMARY_ELIGIBLE",
}
assert frozen_policy["task_eligibility"] == expected_eligibility

for task in TASKS:
    assert (PHASE4D_MODEL_DIR / f"{task}_ridge_confidence_model.joblib").exists()
    assert (PHASE4D_MODEL_DIR / f"{task}_isotonic_calibrator.joblib").exists()

freeze_hashes = {
    str(p.relative_to(ROOT)): sha256_file(p)
    for p in [
        FROZEN_RETRIEVAL_JSON,
        FROZEN_EVIDENCE_JSON,
        FROZEN_GROUNDING_JSON,
        FROZEN_CONFIDENCE_JSON,
        FROZEN_POLICY_JSON,
    ]
}
save_json(freeze_hashes, LOG_DIR / "upstream_freeze_hashes.json")

print("PASS — all frozen Phase 3C–4E artifacts verified.")
print(json.dumps(expected_eligibility, indent=2))

## 5. One-time marker status

If Phase 5 has already been completed, the default behavior is to stop rather than silently
produce a second locked-test analysis.

In [ ]:
if RUN_MARKER.exists():
    marker = load_json(RUN_MARKER)
    print(json.dumps(marker, indent=2))
    if marker.get("status") == "COMPLETE" and not ALLOW_RERUN_AFTER_COMPLETION:
        raise RuntimeError(
            "Phase 5 is already marked COMPLETE. "
            "A repeated locked-test evaluation is disabled by design."
        )
else:
    print("No Phase 5 marker exists yet. Locked test has not been opened by this notebook.")

# STAGE A — Label-blind frozen inference

## 6. Stage A unlock gate and sanitized locked-test view

This is the first cell that opens the locked split. It immediately extracts only:
`id`, `type`, and `body`, then deletes the raw payload. No gold field is retained.

In [ ]:
if not UNLOCK_STAGE_A or STAGE_A_CONFIRMATION != "RUN_FROZEN_LOCKED_TEST_INFERENCE":
    raise RuntimeError(
        "STAGE A IS LOCKED. In Configuration set UNLOCK_STAGE_A=True and "
        'STAGE_A_CONFIRMATION="RUN_FROZEN_LOCKED_TEST_INFERENCE" only when ready.'
    )

if RUN_MARKER.exists():
    marker = load_json(RUN_MARKER)
    if marker.get("status") == "COMPLETE" and not ALLOW_RERUN_AFTER_COMPLETION:
        raise RuntimeError("Phase 5 already completed.")
    print("Resuming existing Phase 5 run.")
else:
    marker = {
        "dataset": "BioASQ14b",
        "phase": "Phase 5",
        "status": "STAGE_A_STARTED",
        "started_utc": datetime.now(timezone.utc).isoformat(),
        "locked_split_sha256": sha256_file(LOCKED_TEST_JSON),
        "method_change_after_unlock_permitted": False,
    }
    save_json(marker, RUN_MARKER)

_locked_payload = load_json(LOCKED_TEST_JSON)
_locked_questions = _locked_payload["questions"]

test_rows = []
for _q in _locked_questions:
    test_rows.append({
        "id": str(_q["id"]),
        "type": str(_q["type"]).strip().lower(),
        "body": str(_q["body"]).strip(),
    })

# Gold-bearing objects are deliberately discarded here.
del _q, _locked_questions, _locked_payload
gc.collect()

test_df = pd.DataFrame(test_rows)
assert len(test_df) == 860
assert test_df["id"].nunique() == 860
assert set(test_df["type"]) == {"yesno", "factoid", "list", "summary"}
assert list(test_df.columns) == ["id", "type", "body"]

SANITIZED_TEST_CSV = STAGEA_DIR / "locked_test_sanitized_questions.csv"
test_df.to_csv(SANITIZED_TEST_CSV, index=False)

print("STAGE A OPENED — label-blind fields only.")
display(test_df["type"].value_counts())
print("Sanitized columns:", test_df.columns.tolist())

## 7. Load the exact frozen closed-corpus order and retrieval indexes

In [ ]:
corpus_df = pd.read_csv(CORPUS_INDEX_CSV, dtype={"pmid": str})
corpus_df["pmid"] = corpus_df["pmid"].astype(str).str.replace(r"\.0$", "", regex=True)
corpus_df["title"] = corpus_df["title"].fillna("").astype(str).str.strip()
corpus_df["abstract"] = corpus_df["abstract"].fillna("").astype(str).str.strip()
corpus_df["text"] = (
    corpus_df["title"] + ". " + corpus_df["abstract"]
).str.strip()

corpus_pmids = corpus_df["pmid"].tolist()
assert len(corpus_pmids) == len(set(corpus_pmids))

article_pmids = [str(x) for x in load_json(ARTICLE_PMIDS_PATH)]
assert article_pmids == corpus_pmids, "MedCPT embedding order != frozen corpus order"

bm25_matrix = sparse.load_npz(BM25_MATRIX_PATH)
vectorizer = joblib.load(BM25_VECTORIZER_PATH)
assert bm25_matrix.shape[0] == len(corpus_df)

article_embeddings = np.load(ARTICLE_EMBED_PATH, mmap_mode="r")
assert article_embeddings.shape[0] == len(corpus_df)

corpus_lookup = (
    corpus_df.set_index("pmid")[["title", "abstract", "text"]]
    .to_dict(orient="index")
)

print("Frozen corpus records:", len(corpus_df))
print("BM25 shape:", bm25_matrix.shape)
print("MedCPT article embeddings:", article_embeddings.shape)

## 8. Frozen BM25 + MedCPT retrieval → RRF Top-100

In [ ]:
BM25_RUN = STAGEA_DIR / "bm25_top100_locked.jsonl"
DENSE_RUN = STAGEA_DIR / "medcpt_top100_locked.jsonl"
RRF_RUN = STAGEA_DIR / "BM25_MedCPT_RRF_top100_locked.jsonl"
QUERY_EMBED_PATH = CACHE_DIR / "medcpt_locked_query_embeddings.npy"

def deterministic_topk(scores, k):
    scores = np.asarray(scores)
    k = min(k, len(scores))
    idx = np.argpartition(-scores, k - 1)[:k]
    # Primary: descending score. Tie-break: original corpus index.
    idx = idx[np.lexsort((idx, -scores[idx]))]
    return idx

if BM25_RUN.exists() and len(read_jsonl(BM25_RUN)) == 860:
    bm25_results = {
        str(r["id"]): [str(x) for x in r["ranked_pmids"]]
        for r in read_jsonl(BM25_RUN)
    }
    print("Loaded cached BM25 run.")
else:
    bm25_results = {}
    for row in tqdm(test_df.itertuples(index=False), total=len(test_df), desc="BM25 locked test"):
        q = vectorizer.transform([row.body]).tocsr()
        if q.nnz > 0:
            q.data[:] = 1.0
        scores = (bm25_matrix @ q.T).toarray().ravel()
        idx = deterministic_topk(scores, RETRIEVAL_TOP_K)
        bm25_results[row.id] = [corpus_pmids[i] for i in idx]
    write_jsonl(
        [{"id": qid, "ranked_pmids": pmids} for qid, pmids in bm25_results.items()],
        BM25_RUN,
    )

if QUERY_EMBED_PATH.exists():
    query_embeddings = np.load(QUERY_EMBED_PATH)
    assert query_embeddings.shape[0] == 860
    print("Loaded cached query embeddings.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("GPU required for MedCPT retrieval.")
    query_tokenizer = AutoTokenizer.from_pretrained(QUERY_MODEL)
    query_model = AutoModel.from_pretrained(QUERY_MODEL).to("cuda").eval()
    parts = []
    queries = test_df["body"].tolist()
    for start in tqdm(range(0, len(queries), DENSE_QUERY_BATCH_SIZE), desc="MedCPT query encoding"):
        batch = queries[start:start + DENSE_QUERY_BATCH_SIZE]
        enc = query_tokenizer(
            batch, truncation=True, padding=True, max_length=QUERY_MAX_LENGTH, return_tensors="pt"
        )
        enc = {k: v.to("cuda") for k, v in enc.items()}
        with torch.no_grad():
            emb = query_model(**enc).last_hidden_state[:, 0, :].float().cpu().numpy()
        parts.append(emb)
    query_embeddings = np.vstack(parts).astype(np.float32)
    np.save(QUERY_EMBED_PATH, query_embeddings)
    del query_model, query_tokenizer
    torch.cuda.empty_cache()
    gc.collect()

if DENSE_RUN.exists() and len(read_jsonl(DENSE_RUN)) == 860:
    dense_results = {
        str(r["id"]): [str(x) for x in r["ranked_pmids"]]
        for r in read_jsonl(DENSE_RUN)
    }
    print("Loaded cached dense run.")
else:
    article_tensor = torch.from_numpy(
        np.asarray(article_embeddings, dtype=np.float32)
    ).to("cuda")
    query_tensor = torch.from_numpy(query_embeddings).to("cuda")
    dense_results = {}
    batch_size = 128
    for start in tqdm(range(0, 860, batch_size), desc="MedCPT inner-product retrieval"):
        end = min(start + batch_size, 860)
        scores = query_tensor[start:end] @ article_tensor.T
        _, top_idx = torch.topk(scores, k=RETRIEVAL_TOP_K, dim=1, largest=True, sorted=True)
        for j, idxs in enumerate(top_idx.cpu().numpy()):
            qid = str(test_df.iloc[start + j]["id"])
            dense_results[qid] = [corpus_pmids[int(i)] for i in idxs]
    write_jsonl(
        [{"id": qid, "ranked_pmids": pmids} for qid, pmids in dense_results.items()],
        DENSE_RUN,
    )
    del article_tensor, query_tensor
    torch.cuda.empty_cache()
    gc.collect()

if RRF_RUN.exists() and len(read_jsonl(RRF_RUN)) == 860:
    rrf_results = {
        str(r["id"]): [str(x) for x in r["ranked_pmids"]]
        for r in read_jsonl(RRF_RUN)
    }
    print("Loaded cached RRF run.")
else:
    rrf_results = {}
    for qid in test_df["id"].astype(str):
        fused = defaultdict(float)
        best_source_rank = {}
        for ranked in [bm25_results[qid], dense_results[qid]]:
            for rank, pmid in enumerate(ranked, start=1):
                fused[pmid] += 1.0 / (RRF_K0 + rank)
                best_source_rank[pmid] = min(best_source_rank.get(pmid, 10**9), rank)
        ordered = sorted(
            fused,
            key=lambda p: (-fused[p], best_source_rank[p], p)
        )[:RETRIEVAL_TOP_K]
        rrf_results[qid] = ordered
    write_jsonl(
        [{"id": qid, "ranked_pmids": pmids} for qid, pmids in rrf_results.items()],
        RRF_RUN,
    )

assert len(rrf_results) == 860
assert all(len(x) == RETRIEVAL_TOP_K for x in rrf_results.values())
print("RRF Top-100 complete:", len(rrf_results))

## 9. Frozen MedCPT Cross-Encoder reranking of RRF Top-100

In [ ]:
RERANK_PAIR_CACHE = CACHE_DIR / "locked_rerank_medcpt_scores.csv"
RERANKED_RUN = STAGEA_DIR / "RRF_MedCPT_CrossEncoder_top100_locked.jsonl"

question_lookup = test_df.set_index("id")["body"].to_dict()

pair_rows = []
for qid in test_df["id"].astype(str):
    for rrf_rank, pmid in enumerate(rrf_results[qid], start=1):
        pair_rows.append({
            "pair_key": f"{qid}|{pmid}",
            "id": qid,
            "pmid": pmid,
            "rrf_rank": rrf_rank,
            "question": question_lookup[qid],
            "article_text": corpus_lookup[pmid]["text"],
        })
rerank_pairs = pd.DataFrame(pair_rows)

if RERANK_PAIR_CACHE.exists():
    rerank_cache = pd.read_csv(RERANK_PAIR_CACHE, dtype={"pair_key": str, "id": str, "pmid": str})
    rerank_cache = rerank_cache.drop_duplicates("pair_key", keep="last")
else:
    rerank_cache = pd.DataFrame(columns=["pair_key", "id", "pmid", "medcpt_score"])

done = set(rerank_cache["pair_key"].astype(str))
todo = rerank_pairs[~rerank_pairs["pair_key"].isin(done)].reset_index(drop=True)
print("Cross-encoder pairs:", len(rerank_pairs), "| remaining:", len(todo))

if len(todo):
    tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(RERANK_MODEL).to("cuda").eval()
    buffer = []
    for start in tqdm(range(0, len(todo), RERANK_BATCH_SIZE), desc="Frozen MedCPT reranking"):
        batch = todo.iloc[start:start + RERANK_BATCH_SIZE]
        enc = tokenizer(
            text=batch["question"].tolist(),
            text_pair=batch["article_text"].tolist(),
            padding=True, truncation=True, max_length=RERANK_MAX_LENGTH,
            return_tensors="pt",
        )
        enc = {k: v.to("cuda") for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits.float()
        if logits.ndim == 1:
            scores = logits
        elif logits.ndim == 2 and logits.shape[1] == 1:
            scores = logits[:, 0]
        elif logits.ndim == 2 and logits.shape[1] == 2:
            scores = logits[:, 1]
        else:
            raise RuntimeError(f"Unexpected reranker logits shape: {tuple(logits.shape)}")
        for rec, score in zip(batch.to_dict("records"), scores.cpu().numpy()):
            buffer.append({
                "pair_key": rec["pair_key"], "id": rec["id"], "pmid": rec["pmid"],
                "medcpt_score": float(score),
            })
        if len(buffer) >= 3000:
            chunk = pd.DataFrame(buffer)
            chunk.to_csv(
                RERANK_PAIR_CACHE, mode="a", header=not RERANK_PAIR_CACHE.exists(), index=False
            )
            buffer = []
    if buffer:
        pd.DataFrame(buffer).to_csv(
            RERANK_PAIR_CACHE, mode="a", header=not RERANK_PAIR_CACHE.exists(), index=False
        )
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

rerank_cache = pd.read_csv(RERANK_PAIR_CACHE, dtype={"pair_key": str, "id": str, "pmid": str})
rerank_cache = rerank_cache.drop_duplicates("pair_key", keep="last")
scored = rerank_pairs.merge(
    rerank_cache[["pair_key", "medcpt_score"]], on="pair_key", how="left", validate="one_to_one"
)
assert scored["medcpt_score"].notna().all()

reranked_results = {}
for qid, g in scored.groupby("id", sort=False):
    g = g.sort_values(
        ["medcpt_score", "rrf_rank", "pmid"],
        ascending=[False, True, True],
        kind="mergesort",
    )
    reranked_results[str(qid)] = g["pmid"].astype(str).tolist()

write_jsonl(
    [{"id": qid, "ranked_pmids": pmids} for qid, pmids in reranked_results.items()],
    RERANKED_RUN,
)
assert len(reranked_results) == 860
assert all(len(v) == 100 for v in reranked_results.values())
print("Frozen reranking complete:", len(reranked_results))

## 10. Frozen Top-5 evidence builder and exact Phase 4B prompt

In [ ]:
SYSTEM_PROMPT = """You are a biomedical question-answering model used in a research benchmark.

Rules:
1. Return JSON only. Do not use Markdown or code fences.
2. Do not provide hidden reasoning or chain-of-thought.
3. Follow the required output shape exactly.
4. The ideal_answer must be concise, factual, and at most 150 words.
5. Do not invent citations or PMID identifiers.
"""

FORMAT_RULES = {
    "yesno": """Question type: yesno.
Return exactly this JSON structure:
{"exact_answer":"yes","ideal_answer":"concise biomedical explanation"}
or use "no" as exact_answer.
The exact_answer must be exactly yes or no.""",

    "factoid": """Question type: factoid.
Return exactly this JSON structure:
{"exact_answer":["answer 1","answer 2"],"ideal_answer":"concise biomedical explanation"}
The exact_answer must contain at most 5 short candidate answers, ordered from most to least likely.""",

    "list": """Question type: list.
Return exactly this JSON structure:
{"exact_answer":["item 1","item 2"],"ideal_answer":"concise biomedical explanation"}
The exact_answer must be a JSON list of distinct short items.""",

    "summary": """Question type: summary.
Return exactly this JSON structure:
{"exact_answer":[],"ideal_answer":"concise biomedical summary"}
The exact_answer must be an empty JSON list."""
}

def build_messages(question, qtype, evidence):
    evidence_rule = (
        "Automatically retrieved PubMed evidence is provided below after "
        "document-level pruning. Base the answer on this evidence and do "
        "not add unsupported biomedical claims."
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": "\n\n".join([
                FORMAT_RULES[qtype],
                evidence_rule,
                f"Question: {question}",
                "Evidence:\n" + evidence,
            ]),
        },
    ]

def llm_token_count(tokenizer, text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def llm_truncate(tokenizer, text, max_tokens):
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return text
    return tokenizer.decode(ids[:max_tokens], skip_special_tokens=True)

def pack_to_budget(tokenizer, blocks, max_tokens):
    packed, used = [], 0
    for block in blocks:
        block = str(block).strip()
        if not block:
            continue
        n = llm_token_count(tokenizer, block)
        if used + n <= max_tokens:
            packed.append(block)
            used += n
        else:
            remaining = max_tokens - used
            if remaining > 32:
                packed.append(llm_truncate(tokenizer, block, remaining))
                used = max_tokens
            break
    return "\n\n".join(packed), used

def build_document_top5_evidence(qid, tokenizer):
    blocks = []
    for rank, pmid in enumerate(reranked_results[str(qid)][:DOCUMENT_TOP_K], start=1):
        rec = corpus_lookup.get(str(pmid))
        if rec is None:
            continue
        blocks.append(
            f"[Filtered document {rank}] PMID: {pmid}\n"
            f"Title: {rec['title']}\n"
            f"Abstract: {rec['abstract']}"
        )
    return pack_to_budget(tokenizer, blocks, MAX_EVIDENCE_TOKENS)

## 11. JSON parser — same frozen answer schema

In [ ]:
def extract_json_object(raw_text):
    text = str(raw_text or "").strip()
    try:
        obj = json.loads(text)
        return obj, "direct_json"
    except Exception:
        pass

    starts = [i for i, c in enumerate(text) if c == "{"]
    for start in starts:
        depth = 0
        in_string = False
        escape = False
        for i in range(start, len(text)):
            c = text[i]
            if in_string:
                if escape:
                    escape = False
                elif c == "\\":
                    escape = True
                elif c == '"':
                    in_string = False
                continue
            if c == '"':
                in_string = True
            elif c == "{":
                depth += 1
            elif c == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[start:i+1]
                    try:
                        return json.loads(candidate), "extracted_json"
                    except Exception:
                        break
    return None, "invalid_json"

def validate_prediction_object(obj, qtype):
    if not isinstance(obj, dict):
        return False, "not_object"
    if "exact_answer" not in obj or "ideal_answer" not in obj:
        return False, "missing_key"
    if not isinstance(obj.get("ideal_answer"), str):
        return False, "ideal_not_string"

    exact = obj.get("exact_answer")
    if qtype == "yesno":
        if not isinstance(exact, str) or exact.strip().lower() not in {"yes", "no"}:
            return False, "yesno_invalid"
    elif qtype in {"factoid", "list"}:
        if not isinstance(exact, list):
            return False, "exact_not_list"
        if not all(isinstance(x, str) for x in exact):
            return False, "exact_list_nonstring"
        if qtype == "factoid" and len(exact) > 5:
            return False, "factoid_more_than_5"
    elif qtype == "summary":
        if not isinstance(exact, list) or len(exact) != 0:
            return False, "summary_exact_not_empty"
    else:
        return False, "unknown_type"
    return True, "valid"

def parse_prediction(raw_text, qtype):
    obj, parse_mode = extract_json_object(raw_text)
    if obj is None:
        return {
            "parse_valid": False,
            "parse_mode": parse_mode,
            "validation_status": "invalid_json",
            "exact_answer_pred": None,
            "ideal_answer_pred": "",
        }
    valid, status = validate_prediction_object(obj, qtype)
    return {
        "parse_valid": bool(valid),
        "parse_mode": parse_mode,
        "validation_status": status,
        "exact_answer_pred": obj.get("exact_answer"),
        "ideal_answer_pred": (
            obj.get("ideal_answer", "")
            if isinstance(obj.get("ideal_answer", ""), str)
            else ""
        ),
    }

## 12. Frozen Qwen generation — resumable, no test labels

In [ ]:
GENERATION_JSONL = STAGEA_DIR / "internal_test_860_document_top5.jsonl"

existing = {}
if GENERATION_JSONL.exists():
    for rec in read_jsonl(GENERATION_JSONL):
        existing[str(rec["id"])] = rec
print("Existing generations:", len(existing))

todo_df = test_df[~test_df["id"].astype(str).isin(set(existing))].reset_index(drop=True)
print("Remaining generations:", len(todo_df))

if len(todo_df):
    if not torch.cuda.is_available():
        raise RuntimeError("GPU required for frozen Qwen generation.")

    qwen_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
    qwen_tokenizer.padding_side = "left"
    if qwen_tokenizer.pad_token_id is None:
        qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

    if USE_4BIT:
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        qwen_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL,
            quantization_config=qconfig,
            device_map="auto",
            torch_dtype=torch.float16,
        ).eval()
    else:
        qwen_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL, device_map="auto", torch_dtype=torch.float16
        ).eval()

    for start in tqdm(range(0, len(todo_df), LLM_BATCH_SIZE), desc="Frozen locked-test Qwen generation"):
        batch_df = todo_df.iloc[start:start + LLM_BATCH_SIZE]
        prompt_texts, metadata = [], []

        for row in batch_df.itertuples(index=False):
            evidence, evidence_tokens = build_document_top5_evidence(row.id, qwen_tokenizer)
            messages = build_messages(row.body, row.type, evidence)
            prompt = qwen_tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            prompt_texts.append(prompt)
            metadata.append((row, evidence_tokens))

        enc = qwen_tokenizer(
            prompt_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        )
        enc = {k: v.to(qwen_model.device) for k, v in enc.items()}
        input_width = enc["input_ids"].shape[1]

        with torch.no_grad():
            out_ids = qwen_model.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=qwen_tokenizer.pad_token_id,
                eos_token_id=qwen_tokenizer.eos_token_id,
            )

        for i, (row, evidence_tokens) in enumerate(metadata):
            new_ids = out_ids[i, input_width:]
            raw_text = qwen_tokenizer.decode(new_ids, skip_special_tokens=True).strip()
            parsed = parse_prediction(raw_text, row.type)
            rec = {
                "id": str(row.id),
                "type": row.type,
                "condition": "document_top5",
                "evidence_tokens": int(evidence_tokens),
                "raw_text": raw_text,
                **parsed,
            }
            append_jsonl(rec, GENERATION_JSONL)
            existing[str(row.id)] = rec

    del qwen_model, qwen_tokenizer
    torch.cuda.empty_cache()
    gc.collect()

generation_df = pd.DataFrame(list(existing.values()))
generation_df["id"] = generation_df["id"].astype(str)
generation_df = generation_df.drop_duplicates("id", keep="last")
assert len(generation_df) == 860
assert generation_df["id"].nunique() == 860
generation_df = test_df[["id", "type"]].merge(
    generation_df, on=["id", "type"], validate="one_to_one"
)
generation_df.to_csv(TABLE_DIR / "locked_test_generation_predictions.csv", index=False)

print("Generation complete:", len(generation_df))
print("JSON valid rate:", generation_df["parse_valid"].mean())
display(generation_df.groupby("type")["parse_valid"].agg(["count", "mean"]))

# STAGE A2 — Frozen Phase 4C v2 grounding

## 13. Deterministic evidence windows and generated claims

In [ ]:
def split_sentences(text):
    text = " ".join(str(text).split()).strip()
    if not text:
        return []
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text)
    return [p.strip() for p in parts if len(p.strip()) >= MIN_SENTENCE_CHARS]

def split_answer_claims(text):
    text = " ".join(str(text).split()).strip()
    if not text:
        return []
    claims = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text)
    clean = [c.strip() for c in claims if len(c.strip()) >= 8]
    return clean if clean else [text]

def build_retrieved_units(qid):
    units = []
    for document_rank, pmid in enumerate(reranked_results[str(qid)][:DOCUMENT_TOP_K], start=1):
        rec = corpus_lookup[str(pmid)]
        title = rec["title"]
        abstract = rec["abstract"]
        sentences = split_sentences(abstract)
        if not sentences and len(abstract) >= MIN_SENTENCE_CHARS:
            sentences = [abstract]

        windows = []
        if len(sentences) <= EVIDENCE_WINDOW_SENTENCES:
            if sentences:
                windows = [" ".join(sentences)]
        else:
            for i in range(0, len(sentences) - EVIDENCE_WINDOW_SENTENCES + 1, EVIDENCE_WINDOW_STRIDE):
                windows.append(" ".join(sentences[i:i + EVIDENCE_WINDOW_SENTENCES]))
        windows = windows[:MAX_WINDOWS_PER_DOCUMENT]

        for window_index, window in enumerate(windows, start=1):
            text = f"Title: {title}\nEvidence: {window}" if title else f"Evidence: {window}"
            units.append({
                "id": str(qid),
                "condition": "document_top5",
                "evidence_id": f"document_top5|{qid}|{pmid}|{document_rank}|{window_index}",
                "pmid": str(pmid),
                "document_rank": int(document_rank),
                "window_index": int(window_index),
                "evidence_text": text,
            })
    return units

evidence_rows = []
for qid in tqdm(test_df["id"].astype(str), desc="Building locked-test evidence units"):
    evidence_rows.extend(build_retrieved_units(qid))
evidence_units_df = pd.DataFrame(evidence_rows)
assert evidence_units_df["evidence_id"].is_unique
evidence_units_df.to_csv(STAGEA_DIR / "grounding_evidence_units.csv", index=False)

claim_rows = []
for row in generation_df.itertuples(index=False):
    ideal = str(row.ideal_answer_pred or "").strip() if bool(row.parse_valid) else ""
    for claim_index, claim in enumerate(split_answer_claims(ideal), start=1):
        claim_rows.append({
            "id": str(row.id),
            "type": row.type,
            "condition": "document_top5",
            "parse_valid": bool(row.parse_valid),
            "claim_index": int(claim_index),
            "claim_id": f"document_top5|{row.id}|{claim_index}",
            "claim_text": claim,
        })
claims_df = pd.DataFrame(claim_rows)
claims_df.to_csv(STAGEA_DIR / "generated_answer_claims.csv", index=False)

print("Evidence units:", len(evidence_units_df))
print("Generated claims:", len(claims_df))

## 14. Build claim × evidence pairs

In [ ]:
evidence_group_lookup = {
    key: group.copy()
    for key, group in evidence_units_df.groupby(["id", "condition"], sort=False)
}

pair_rows = []
for claim in tqdm(claims_df.itertuples(index=False), total=len(claims_df), desc="Claim-evidence pairs"):
    evidence = evidence_group_lookup.get((str(claim.id), "document_top5"))
    if evidence is None or len(evidence) == 0:
        continue
    for e in evidence.itertuples(index=False):
        raw = f"{claim.claim_id}|{e.evidence_id}"
        pair_key = hashlib.sha256(raw.encode("utf-8")).hexdigest()
        pair_rows.append({
            "pair_key": pair_key,
            "claim_id": claim.claim_id,
            "id": str(claim.id),
            "type": claim.type,
            "claim_text": claim.claim_text,
            "evidence_id": e.evidence_id,
            "pmid": str(e.pmid),
            "document_rank": int(e.document_rank),
            "window_index": int(e.window_index),
            "evidence_text": e.evidence_text,
        })
grounding_pairs = pd.DataFrame(pair_rows)
assert grounding_pairs["pair_key"].is_unique
print("Grounding relevance pairs:", len(grounding_pairs))

## 15. Frozen MedCPT claim–evidence relevance scoring and fixed Top-3

In [ ]:
GROUNDING_MEDCPT_CACHE = CACHE_DIR / "locked_grounding_medcpt_scores.csv"

if GROUNDING_MEDCPT_CACHE.exists():
    cache = pd.read_csv(GROUNDING_MEDCPT_CACHE, dtype={"pair_key": str})
    cache = cache.drop_duplicates("pair_key", keep="last")
else:
    cache = pd.DataFrame(columns=["pair_key", "medcpt_score"])

done = set(cache["pair_key"].astype(str))
todo = grounding_pairs[~grounding_pairs["pair_key"].isin(done)].reset_index(drop=True)
print("Grounding MedCPT remaining:", len(todo))

if len(todo):
    tok = AutoTokenizer.from_pretrained(GROUNDING_MEDCPT_MODEL)
    mod = AutoModelForSequenceClassification.from_pretrained(
        GROUNDING_MEDCPT_MODEL
    ).to("cuda").eval()
    buffer = []
    for start in tqdm(
        range(0, len(todo), GROUNDING_MEDCPT_BATCH_SIZE),
        desc="MedCPT claim-evidence relevance",
    ):
        batch = todo.iloc[start:start + GROUNDING_MEDCPT_BATCH_SIZE]
        enc = tok(
            text=batch["claim_text"].tolist(),
            text_pair=batch["evidence_text"].tolist(),
            padding=True, truncation=True, max_length=GROUNDING_MEDCPT_MAX_LENGTH,
            return_tensors="pt",
        )
        enc = {k: v.to("cuda") for k, v in enc.items()}
        with torch.no_grad():
            logits = mod(**enc).logits.float()
        if logits.ndim == 1:
            scores = logits
        elif logits.ndim == 2 and logits.shape[1] == 1:
            scores = logits[:, 0]
        elif logits.ndim == 2 and logits.shape[1] == 2:
            scores = logits[:, 1]
        else:
            raise RuntimeError(f"Unexpected MedCPT logits shape: {tuple(logits.shape)}")
        for key, score in zip(batch["pair_key"], scores.cpu().numpy()):
            buffer.append({"pair_key": str(key), "medcpt_score": float(score)})
        if len(buffer) >= 3000:
            pd.DataFrame(buffer).to_csv(
                GROUNDING_MEDCPT_CACHE, mode="a",
                header=not GROUNDING_MEDCPT_CACHE.exists(), index=False
            )
            buffer = []
    if buffer:
        pd.DataFrame(buffer).to_csv(
            GROUNDING_MEDCPT_CACHE, mode="a",
            header=not GROUNDING_MEDCPT_CACHE.exists(), index=False
        )
    del mod, tok
    torch.cuda.empty_cache()
    gc.collect()

cache = pd.read_csv(GROUNDING_MEDCPT_CACHE, dtype={"pair_key": str}).drop_duplicates("pair_key", keep="last")
scored_pairs = grounding_pairs.merge(cache, on="pair_key", how="left", validate="one_to_one")
assert scored_pairs["medcpt_score"].notna().all()

parts = []
for claim_id, g in scored_pairs.groupby("claim_id", sort=False):
    g = g.copy()
    g["document_rank_sort"] = g["document_rank"].fillna(10000)
    g = (
        g.sort_values(
            ["medcpt_score", "document_rank_sort", "window_index", "evidence_id"],
            ascending=[False, True, True, True],
            kind="mergesort",
        )
        .head(GROUNDING_TOP_M)
        .reset_index(drop=True)
    )
    g["medcpt_rank"] = np.arange(len(g)) + 1
    parts.append(g.drop(columns=["document_rank_sort"]))

selected_top3 = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
selected_top3.to_csv(STAGEA_DIR / "selected_medcpt_top3_per_claim.csv", index=False)
print("Selected Top-3 rows:", len(selected_top3))

## 16. Frozen biomedical NLI on selected Top-3

In [ ]:
NLI_SCORE_CACHE = CACHE_DIR / "locked_top3_claim_evidence_nli_scores.csv"

if NLI_SCORE_CACHE.exists():
    nli_cache = pd.read_csv(NLI_SCORE_CACHE, dtype={"pair_key": str}).drop_duplicates("pair_key", keep="last")
else:
    nli_cache = pd.DataFrame(columns=[
        "pair_key", "p_contradiction", "p_entailment", "p_neutral", "nli_label"
    ])

done = set(nli_cache["pair_key"].astype(str))
todo = selected_top3[~selected_top3["pair_key"].isin(done)].reset_index(drop=True)
print("NLI remaining:", len(todo))

if len(todo):
    nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
    nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to("cuda").eval()

    raw_id2label = {
        int(k): str(v).lower()
        for k, v in nli_model.config.id2label.items()
    }
    label2id = {v: k for k, v in raw_id2label.items()}
    if not {"contradiction", "entailment", "neutral"}.issubset(label2id):
        raise RuntimeError(f"Unexpected NLI labels: {raw_id2label}")

    CONTRADICTION_IDX = label2id["contradiction"]
    ENTAILMENT_IDX = label2id["entailment"]
    NEUTRAL_IDX = label2id["neutral"]

    buffer = []
    for start in tqdm(range(0, len(todo), NLI_BATCH_SIZE), desc="Biomedical NLI on MedCPT Top-3"):
        batch = todo.iloc[start:start + NLI_BATCH_SIZE]
        enc = nli_tokenizer(
            text=batch["evidence_text"].tolist(),
            text_pair=batch["claim_text"].tolist(),
            padding=True, truncation=True, max_length=NLI_MAX_LENGTH,
            return_tensors="pt",
        )
        enc = {k: v.to("cuda") for k, v in enc.items()}
        with torch.no_grad():
            logits = nli_model(**enc).logits.float()
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        pred = probs.argmax(axis=1)

        for i, key in enumerate(batch["pair_key"]):
            buffer.append({
                "pair_key": str(key),
                "p_contradiction": float(probs[i, CONTRADICTION_IDX]),
                "p_entailment": float(probs[i, ENTAILMENT_IDX]),
                "p_neutral": float(probs[i, NEUTRAL_IDX]),
                "nli_label": raw_id2label[int(pred[i])],
            })
        if len(buffer) >= 3000:
            pd.DataFrame(buffer).to_csv(
                NLI_SCORE_CACHE, mode="a",
                header=not NLI_SCORE_CACHE.exists(), index=False
            )
            buffer = []
    if buffer:
        pd.DataFrame(buffer).to_csv(
            NLI_SCORE_CACHE, mode="a",
            header=not NLI_SCORE_CACHE.exists(), index=False
        )

    del nli_model, nli_tokenizer
    torch.cuda.empty_cache()
    gc.collect()

nli_cache = pd.read_csv(NLI_SCORE_CACHE, dtype={"pair_key": str}).drop_duplicates("pair_key", keep="last")
top3_nli = selected_top3.merge(nli_cache, on="pair_key", how="left", validate="one_to_one")
assert top3_nli["p_entailment"].notna().all()
top3_nli.to_csv(STAGEA_DIR / "top3_claim_evidence_nli.csv", index=False)
print("NLI-scored Top-3 pairs:", len(top3_nli))

## 17. Claim-level and question-level grounding features

In [ ]:
claim_ground_rows = []

for claim_id, g in top3_nli.groupby("claim_id", sort=False):
    g = g.sort_values("medcpt_rank")
    top1 = g.iloc[0]
    claim_ground_rows.append({
        "claim_id": claim_id,
        "id": str(top1["id"]),
        "type": top1["type"],
        "condition": "document_top5",
        "claim_text": top1["claim_text"],
        "top1_medcpt_score": float(top1["medcpt_score"]),
        "top1_document_rank": float(top1["document_rank"]),
        "top1_entailment": float(top1["p_entailment"]),
        "top1_neutral": float(top1["p_neutral"]),
        "top1_contradiction": float(top1["p_contradiction"]),
        "top1_nli_label": str(top1["nli_label"]),
        "top1_supported": float(str(top1["nli_label"]) == "entailment"),
        "top1_contradiction_flag": float(str(top1["nli_label"]) == "contradiction"),
        "top3_max_entailment": float(g["p_entailment"].max()),
        "top3_supported": float((g["nli_label"].astype(str) == "entailment").any()),
    })

claim_grounding_df = pd.DataFrame(claim_ground_rows)
claim_grounding_df.to_csv(STAGEA_DIR / "claim_grounding_v2.csv", index=False)

question_rows = []
for row in test_df.itertuples(index=False):
    g = claim_grounding_df[claim_grounding_df["id"].astype(str) == str(row.id)]
    if len(g) == 0:
        question_rows.append({
            "id": str(row.id), "type": row.type, "condition": "document_top5",
            "n_claims": 0,
            "mean_top1_entailment": 0.0,
            "mean_top1_contradiction": 0.0,
            "top1_supported_claim_rate": 0.0,
            "top1_contradiction_rate": 0.0,
            "mean_top3_max_entailment": 0.0,
            "top3_supported_claim_rate": 0.0,
            "fully_top1_supported": 0.0,
            "fully_top3_supported": 0.0,
            "mean_top1_medcpt_score": 0.0,
            "min_top1_medcpt_score": 0.0,
            "mean_top1_document_rank": 0.0,
            "rank1_document_fraction": 0.0,
            "min_top1_entailment_claim": 0.0,
            "max_top1_contradiction_claim": 0.0,
        })
        continue

    question_rows.append({
        "id": str(row.id), "type": row.type, "condition": "document_top5",
        "n_claims": int(len(g)),
        "mean_top1_entailment": float(g["top1_entailment"].mean()),
        "mean_top1_contradiction": float(g["top1_contradiction"].mean()),
        "top1_supported_claim_rate": float(g["top1_supported"].mean()),
        "top1_contradiction_rate": float(g["top1_contradiction_flag"].mean()),
        "mean_top3_max_entailment": float(g["top3_max_entailment"].mean()),
        "top3_supported_claim_rate": float(g["top3_supported"].mean()),
        "fully_top1_supported": float(g["top1_supported"].min() == 1.0),
        "fully_top3_supported": float(g["top3_supported"].min() == 1.0),
        "mean_top1_medcpt_score": float(g["top1_medcpt_score"].mean()),
        "min_top1_medcpt_score": float(g["top1_medcpt_score"].min()),
        "mean_top1_document_rank": float(g["top1_document_rank"].mean()),
        "rank1_document_fraction": float((g["top1_document_rank"] == 1).mean()),
        "min_top1_entailment_claim": float(g["top1_entailment"].min()),
        "max_top1_contradiction_claim": float(g["top1_contradiction"].max()),
    })

question_grounding_df = pd.DataFrame(question_rows)
assert len(question_grounding_df) == 860
question_grounding_df.to_csv(STAGEA_DIR / "question_grounding_v2.csv", index=False)

display(
    question_grounding_df.groupby("type")[
        ["mean_top1_entailment", "top1_supported_claim_rate", "top1_contradiction_rate"]
    ].mean().round(4)
)

# STAGE A3 — Apply frozen task-specific confidence and policies

## 18. Build the exact inference-time confidence feature table

In [ ]:
NUMERIC_FEATURES = [
    "question_words",
    "ideal_answer_words",
    "parse_valid",
    "n_claims",
    "mean_top1_entailment",
    "mean_top1_contradiction",
    "top1_supported_claim_rate",
    "top1_contradiction_rate",
    "mean_top3_max_entailment",
    "top3_supported_claim_rate",
    "fully_top1_supported",
    "fully_top3_supported",
    "mean_top1_medcpt_score",
    "min_top1_medcpt_score",
    "mean_top1_document_rank",
    "rank1_document_fraction",
    "min_top1_entailment_claim",
    "max_top1_contradiction_claim",
]

generation_features = (
    generation_df[["id", "type", "ideal_answer_pred", "parse_valid"]]
    .merge(test_df[["id", "body"]], on="id", validate="one_to_one")
)
generation_features["question_words"] = generation_features["body"].fillna("").astype(str).str.split().str.len()
generation_features["ideal_answer_words"] = generation_features["ideal_answer_pred"].fillna("").astype(str).str.split().str.len()
generation_features["parse_valid"] = generation_features["parse_valid"].astype(int)

feature_df = generation_features[
    ["id", "type", "question_words", "ideal_answer_words", "parse_valid"]
].merge(
    question_grounding_df.drop(columns=["condition"]),
    on=["id", "type"],
    validate="one_to_one",
)

for col in NUMERIC_FEATURES:
    if col not in feature_df.columns:
        raise KeyError(f"Missing frozen confidence feature: {col}")
    feature_df[col] = pd.to_numeric(feature_df[col], errors="coerce").fillna(0.0)

assert len(feature_df) == 860
feature_df.to_csv(STAGEA_DIR / "locked_test_confidence_features.csv", index=False)
display(feature_df.groupby("type")[["question_words", "ideal_answer_words", "n_claims"]].describe())

## 19. Apply already-fitted task-specific Ridge + isotonic models

In [ ]:
confidence_parts = []

for task in TASKS:
    part = feature_df[feature_df["type"] == task].copy()
    ridge = joblib.load(PHASE4D_MODEL_DIR / f"{task}_ridge_confidence_model.joblib")
    isotonic = joblib.load(PHASE4D_MODEL_DIR / f"{task}_isotonic_calibrator.joblib")

    raw = np.asarray(ridge.predict(part[NUMERIC_FEATURES]), dtype=float)
    calibrated = np.asarray(isotonic.predict(raw), dtype=float)

    part[RANKING_SCORE] = raw
    part[CALIBRATED_CONFIDENCE] = np.clip(calibrated, 0.0, 1.0)
    confidence_parts.append(part)

confidence_df = pd.concat(confidence_parts, ignore_index=True)
confidence_df = test_df[["id", "type"]].merge(
    confidence_df, on=["id", "type"], validate="one_to_one"
)
assert len(confidence_df) == 860

CONFIDENCE_CSV = STAGEA_DIR / "locked_test_frozen_confidence_predictions.csv"
confidence_df.to_csv(CONFIDENCE_CSV, index=False)

display(
    confidence_df.groupby("type")[[RANKING_SCORE, CALIBRATED_CONFIDENCE]]
    .agg(["count", "mean", "std", "min", "max"])
    .round(4)
)

## 20. Freeze Policy A / Policy B selection masks before reading gold

In [ ]:
def select_top_fraction(frame, coverage, score_col=RANKING_SCORE):
    assert 0 < coverage <= 1
    frame = (
        frame.copy()
        .sort_values(score_col, ascending=False, kind="mergesort")
        .reset_index(drop=True)
    )
    k = int(math.ceil(float(coverage) * len(frame)))
    return frame.iloc[:k].copy(), frame.iloc[k:].copy()

def make_policy_mask(frame, policy):
    rows = []
    for task in TASKS:
        task_df = frame[frame["type"] == task].copy()
        coverage = float(policy["task_coverages"][task])
        if coverage >= 1.0:
            selected = task_df.copy()
        else:
            selected, _ = select_top_fraction(task_df, coverage)
        selected_ids = set(selected["id"].astype(str))
        for r in task_df.itertuples(index=False):
            rows.append({
                "id": str(r.id),
                "type": task,
                "policy": policy["name"],
                "target_task_coverage": coverage,
                "selected_to_answer": bool(str(r.id) in selected_ids),
                RANKING_SCORE: float(getattr(r, RANKING_SCORE)),
                CALIBRATED_CONFIDENCE: float(getattr(r, CALIBRATED_CONFIDENCE)),
            })
    return pd.DataFrame(rows)

policy_a_mask = make_policy_mask(confidence_df, POLICY_A)
policy_b_mask = make_policy_mask(confidence_df, POLICY_B)

POLICY_MASK_CSV = STAGEA_DIR / "frozen_policy_selection_masks.csv"
policy_masks = pd.concat([policy_a_mask, policy_b_mask], ignore_index=True)
policy_masks.to_csv(POLICY_MASK_CSV, index=False)

mask_summary = (
    policy_masks.groupby(["policy", "type"])
    .agg(n_total=("id", "size"), n_answered=("selected_to_answer", "sum"))
    .reset_index()
)
mask_summary["coverage"] = mask_summary["n_answered"] / mask_summary["n_total"]
display(mask_summary)

print("Policy masks frozen WITHOUT locked-test gold labels.")

## 21. Freeze and hash Stage A

After this cell, question ranking and abstention choices are immutable.
Stage B will refuse to score if any listed Stage A artifact changes.

In [ ]:
stagea_artifacts = [
    SANITIZED_TEST_CSV,
    BM25_RUN,
    DENSE_RUN,
    RRF_RUN,
    RERANKED_RUN,
    GENERATION_JSONL,
    STAGEA_DIR / "grounding_evidence_units.csv",
    STAGEA_DIR / "generated_answer_claims.csv",
    STAGEA_DIR / "selected_medcpt_top3_per_claim.csv",
    STAGEA_DIR / "top3_claim_evidence_nli.csv",
    STAGEA_DIR / "claim_grounding_v2.csv",
    STAGEA_DIR / "question_grounding_v2.csv",
    STAGEA_DIR / "locked_test_confidence_features.csv",
    CONFIDENCE_CSV,
    POLICY_MASK_CSV,
]

for p in stagea_artifacts:
    assert p.exists(), p

manifest = {
    "dataset": "BioASQ14b",
    "n_locked_test": 860,
    "stage": "A_label_blind_complete",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "gold_scoring_performed": False,
    "ranking_signal": RANKING_SCORE,
    "reported_confidence": CALIBRATED_CONFIDENCE,
    "primary_policy": POLICY_A,
    "secondary_policy": POLICY_B,
    "locked_split_sha256": sha256_file(LOCKED_TEST_JSON),
    "upstream_freeze_hashes": freeze_hashes,
    "stageA_artifact_hashes": {
        str(p.relative_to(PHASE5_DIR)): sha256_file(p)
        for p in stagea_artifacts
    },
}
save_json(manifest, STAGEA_MANIFEST)

marker = load_json(RUN_MARKER)
marker["status"] = "STAGE_A_FROZEN"
marker["stageA_frozen_utc"] = datetime.now(timezone.utc).isoformat()
marker["stageA_manifest_sha256"] = sha256_file(STAGEA_MANIFEST)
save_json(marker, RUN_MARKER)

print("STAGE A FROZEN.")
print("Manifest:", STAGEA_MANIFEST)
print("Manifest SHA256:", sha256_file(STAGEA_MANIFEST))

# STOP POINT 5A — Do not change the pipeline

At this point the following are fixed **before gold scoring**:

- all 860 generated answers;
- all reranked Top-100 / Top-5 evidence choices;
- Phase 4C v2 grounding outputs;
- raw Ridge rankings;
- isotonic reported confidence;
- Policy A and Policy B answer/abstain masks.

No output above should be used to redesign the method.  
To perform the single final scoring pass, enable the Stage B gate in the configuration cell.

# STAGE B — One-time locked gold evaluation

## 22. Stage B gate and integrity verification

In [ ]:
if not OPEN_GOLD_STAGE_B or STAGE_B_CONFIRMATION != "SCORE_FROZEN_LOCKED_TEST_ONCE":
    raise RuntimeError(
        "STAGE B IS LOCKED. Set OPEN_GOLD_STAGE_B=True and "
        'STAGE_B_CONFIRMATION="SCORE_FROZEN_LOCKED_TEST_ONCE" only after Stage A is frozen.'
    )

assert STAGEA_MANIFEST.exists(), "Stage A manifest missing."
manifest = load_json(STAGEA_MANIFEST)
assert manifest["gold_scoring_performed"] is False
assert manifest["locked_split_sha256"] == sha256_file(LOCKED_TEST_JSON)

for rel, expected_hash in manifest["stageA_artifact_hashes"].items():
    p = PHASE5_DIR / rel
    actual = sha256_file(p)
    assert actual == expected_hash, f"Stage A artifact changed: {p}"

current_freeze_hashes = {
    str(p.relative_to(ROOT)): sha256_file(p)
    for p in [
        FROZEN_RETRIEVAL_JSON,
        FROZEN_EVIDENCE_JSON,
        FROZEN_GROUNDING_JSON,
        FROZEN_CONFIDENCE_JSON,
        FROZEN_POLICY_JSON,
    ]
}
assert current_freeze_hashes == manifest["upstream_freeze_hashes"], "An upstream freeze file changed."

marker = load_json(RUN_MARKER)
assert marker["status"] == "STAGE_A_FROZEN", marker
marker["status"] = "STAGE_B_GOLD_OPENED"
marker["stageB_gold_opened_utc"] = datetime.now(timezone.utc).isoformat()
save_json(marker, RUN_MARKER)

print("PASS — Stage A hashes verified. Gold scoring is now opened once.")

## 23. Load locked-test gold labels/documents for final scoring only

In [ ]:
payload = load_json(LOCKED_TEST_JSON)
questions = payload["questions"]

gold_rows = []
for q in questions:
    gold_rows.append({
        "id": str(q["id"]),
        "type": str(q["type"]).strip().lower(),
        "exact_answer_gold": q.get("exact_answer"),
        "ideal_answer_gold": q.get("ideal_answer"),
        "documents_gold": q.get("documents", []),
    })
gold_df = pd.DataFrame(gold_rows)
del q, questions, payload
gc.collect()

assert len(gold_df) == 860
assert set(gold_df["id"]) == set(test_df["id"])
assert gold_df[["id", "type"]].equals(
    test_df[["id", "type"]].sort_values("id").reset_index(drop=True)
) is False or True
# Strong one-to-one type consistency check:
_check = test_df[["id", "type"]].merge(
    gold_df[["id", "type"]], on="id", suffixes=("_sanitized", "_gold"), validate="one_to_one"
)
assert (_check["type_sanitized"] == _check["type_gold"]).all()
del _check

print("Gold labels loaded for final Stage B scoring:", len(gold_df))
display(gold_df["type"].value_counts())

## 24. Frozen answer-quality scoring helpers

In [ ]:
def normalize_answer_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower().strip()
    text = re.sub(r"[_]+", " ", text)
    text = re.sub(r"[^\w\s\.\-\+/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def as_gold_groups(exact_answer, qtype):
    if qtype == "yesno":
        return [[normalize_answer_text(exact_answer)]] if exact_answer is not None else []
    if qtype not in {"factoid", "list"} or exact_answer is None:
        return []
    if not isinstance(exact_answer, list):
        exact_answer = [exact_answer]
    groups = []
    for item in exact_answer:
        if isinstance(item, list):
            syns = [normalize_answer_text(x) for x in item if str(x).strip()]
        else:
            syns = [normalize_answer_text(item)]
        syns = [x for x in syns if x]
        if syns:
            groups.append(syns)
    return groups

def prediction_list(pred):
    if isinstance(pred, list):
        return [str(x).strip() for x in pred if str(x).strip()]
    if isinstance(pred, str) and pred.strip():
        return [pred.strip()]
    return []

def score_factoid(pred, gold):
    preds = prediction_list(pred)[:5]
    groups = as_gold_groups(gold, "factoid")
    all_gold = {syn for group in groups for syn in group}
    norm_preds = [normalize_answer_text(x) for x in preds]
    strict = float(bool(norm_preds) and norm_preds[0] in all_gold)
    first_rank = None
    for rank, p in enumerate(norm_preds, start=1):
        if p in all_gold:
            first_rank = rank
            break
    return {
        "factoid_strict": strict,
        "factoid_lenient": float(first_rank is not None),
        "factoid_mrr": 1.0 / first_rank if first_rank else 0.0,
    }

def score_list(pred, gold):
    preds = prediction_list(pred)
    groups = as_gold_groups(gold, "list")
    unmatched = set(range(len(groups)))
    matched = 0
    for pred_item in preds:
        p = normalize_answer_text(pred_item)
        for gi in list(unmatched):
            if p in set(groups[gi]):
                matched += 1
                unmatched.remove(gi)
                break
    precision = matched / len(preds) if preds else 0.0
    recall = matched / len(groups) if groups else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return {"list_precision": precision, "list_recall": recall, "list_f1": f1}

rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def get_gold_ideal(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return str(value[0]).strip() if value else ""
    return str(value).strip()

def score_ideal(pred, gold, parse_valid=True):
    if not parse_valid:
        return {"rouge1_f": 0.0, "rouge2_f": 0.0, "rougeL_f": 0.0}
    pred = str(pred or "").strip()
    gold = get_gold_ideal(gold)
    if not pred or not gold:
        return {"rouge1_f": 0.0, "rouge2_f": 0.0, "rougeL_f": 0.0}
    s = rouge.score(gold, pred)
    return {
        "rouge1_f": s["rouge1"].fmeasure,
        "rouge2_f": s["rouge2"].fmeasure,
        "rougeL_f": s["rougeL"].fmeasure,
    }

## 25. Score all 860 frozen generated answers

In [ ]:
gold_lookup = gold_df.set_index("id").to_dict(orient="index")

scored_rows = []
for row in generation_df.itertuples(index=False):
    gold = gold_lookup[str(row.id)]
    qtype = row.type
    valid = bool(row.parse_valid)

    rec = {
        "id": str(row.id), "type": qtype, "condition": "document_top5",
        "parse_valid": valid,
        "yesno_correct": np.nan,
        "factoid_strict": np.nan,
        "factoid_lenient": np.nan,
        "factoid_mrr": np.nan,
        "list_precision": np.nan,
        "list_recall": np.nan,
        "list_f1": np.nan,
    }

    if qtype == "yesno":
        rec["yesno_correct"] = (
            float(normalize_answer_text(row.exact_answer_pred) ==
                  normalize_answer_text(gold["exact_answer_gold"]))
            if valid else 0.0
        )
    elif qtype == "factoid":
        rec.update(
            score_factoid(row.exact_answer_pred, gold["exact_answer_gold"])
            if valid else
            {"factoid_strict": 0.0, "factoid_lenient": 0.0, "factoid_mrr": 0.0}
        )
    elif qtype == "list":
        rec.update(
            score_list(row.exact_answer_pred, gold["exact_answer_gold"])
            if valid else
            {"list_precision": 0.0, "list_recall": 0.0, "list_f1": 0.0}
        )

    rec.update(score_ideal(
        row.ideal_answer_pred,
        gold["ideal_answer_gold"],
        parse_valid=valid,
    ))
    scored_rows.append(rec)

score_df = pd.DataFrame(scored_rows)

def task_score_from_row(row):
    if row["type"] == "yesno":
        return float(row["yesno_correct"])
    if row["type"] == "factoid":
        return float(row["factoid_mrr"])
    if row["type"] == "list":
        return float(row["list_f1"])
    if row["type"] == "summary":
        return float(row["rougeL_f"])
    raise ValueError(row["type"])

score_df["task_score"] = score_df.apply(task_score_from_row, axis=1)
assert score_df["task_score"].between(0, 1).all()
score_df.to_csv(TABLE_DIR / "locked_test_answer_scores_per_question.csv", index=False)

yes = score_df[score_df["type"] == "yesno"]
raw_gen = generation_df.set_index("id")
y_true, y_pred = [], []
for qid in yes["id"]:
    y_true.append(normalize_answer_text(gold_lookup[str(qid)]["exact_answer_gold"]))
    rr = raw_gen.loc[str(qid)]
    y_pred.append(
        normalize_answer_text(rr["exact_answer_pred"]) if bool(rr["parse_valid"]) else "__invalid__"
    )

summary = {
    "n": 860,
    "json_valid_rate": float(score_df["parse_valid"].mean()),
    "rouge1_f": float(score_df["rouge1_f"].mean()),
    "rouge2_f": float(score_df["rouge2_f"].mean()),
    "rougeL_f": float(score_df["rougeL_f"].mean()),
    "yesno_accuracy": float(yes["yesno_correct"].mean()),
    "yesno_macro_f1": float(f1_score(
        y_true, y_pred, labels=["yes", "no"], average="macro", zero_division=0
    )),
    "factoid_strict": float(score_df.loc[score_df["type"]=="factoid", "factoid_strict"].mean()),
    "factoid_lenient": float(score_df.loc[score_df["type"]=="factoid", "factoid_lenient"].mean()),
    "factoid_mrr": float(score_df.loc[score_df["type"]=="factoid", "factoid_mrr"].mean()),
    "list_precision": float(score_df.loc[score_df["type"]=="list", "list_precision"].mean()),
    "list_recall": float(score_df.loc[score_df["type"]=="list", "list_recall"].mean()),
    "list_f1": float(score_df.loc[score_df["type"]=="list", "list_f1"].mean()),
    "summary_rougeL_f": float(score_df.loc[score_df["type"]=="summary", "rougeL_f"].mean()),
}
summary["macro_task_score"] = float(np.mean([
    summary["yesno_accuracy"],
    summary["factoid_mrr"],
    summary["list_f1"],
    summary["summary_rougeL_f"],
]))

answer_summary_df = pd.DataFrame([summary])
answer_summary_df.to_csv(TABLE_DIR / "locked_test_answer_summary.csv", index=False)
display(answer_summary_df.T)

## 26. Final locked-test retrieval metrics

In [ ]:
def extract_pmid(ref):
    s = str(ref).strip().rstrip("/")
    m = re.search(r"(\d+)(?:\D*)$", s)
    return m.group(1) if m else None

gold_docs_by_qid = {}
for row in gold_df.itertuples(index=False):
    docs = {extract_pmid(x) for x in row.documents_gold}
    docs.discard(None)
    gold_docs_by_qid[str(row.id)] = docs

def retrieval_metrics_at_k(ranked, gold, k=10):
    gold = set(gold)
    pred = ranked[:k]
    rel = np.array([1 if p in gold else 0 for p in pred], dtype=float)
    hits = int(rel.sum())
    recall = hits / len(gold) if gold else np.nan
    precision = hits / k
    hitrate = float(hits > 0)

    precisions = []
    hit_count = 0
    first_rank = None
    dcg = 0.0
    for i, r in enumerate(rel, start=1):
        if r:
            hit_count += 1
            precisions.append(hit_count / i)
            if first_rank is None:
                first_rank = i
            dcg += 1.0 / math.log2(i + 1)
    ap = sum(precisions) / min(len(gold), k) if gold else np.nan
    mrr = 1.0 / first_rank if first_rank else 0.0
    ideal_hits = min(len(gold), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    ndcg = dcg / idcg if idcg > 0 else np.nan
    return precision, recall, hitrate, ap, mrr, ndcg

retr_rows = []
for qid in test_df["id"].astype(str):
    p, r, h, ap, mrr, ndcg = retrieval_metrics_at_k(
        reranked_results[qid], gold_docs_by_qid[qid], k=10
    )
    retr_rows.append({
        "id": qid, "precision@10": p, "recall@10": r, "hitrate@10": h,
        "MAP@10": ap, "MRR@10": mrr, "nDCG@10": ndcg,
    })
retrieval_test_df = pd.DataFrame(retr_rows)
retrieval_test_df.to_csv(TABLE_DIR / "locked_test_retrieval_per_question.csv", index=False)

retrieval_summary = {
    "n_queries": 860,
    **{c: float(retrieval_test_df[c].mean()) for c in
       ["precision@10", "recall@10", "hitrate@10", "MAP@10", "MRR@10", "nDCG@10"]}
}
display(pd.DataFrame([retrieval_summary]))

## 27. Locked-test grounding summary

In [ ]:
grounding_summary = (
    question_grounding_df.groupby("type")
    .agg(
        n=("id", "size"),
        mean_top1_entailment=("mean_top1_entailment", "mean"),
        top1_supported_claim_rate=("top1_supported_claim_rate", "mean"),
        fully_top1_supported_answer_rate=("fully_top1_supported", "mean"),
        top1_contradiction_rate=("top1_contradiction_rate", "mean"),
        mean_top3_max_entailment=("mean_top3_max_entailment", "mean"),
        top3_supported_claim_rate=("top3_supported_claim_rate", "mean"),
        fully_top3_supported_answer_rate=("fully_top3_supported", "mean"),
    )
    .reset_index()
)
overall_grounding = pd.DataFrame([{
    "type": "ALL",
    "n": 860,
    "mean_top1_entailment": float(question_grounding_df["mean_top1_entailment"].mean()),
    "top1_supported_claim_rate": float(question_grounding_df["top1_supported_claim_rate"].mean()),
    "fully_top1_supported_answer_rate": float(question_grounding_df["fully_top1_supported"].mean()),
    "top1_contradiction_rate": float(question_grounding_df["top1_contradiction_rate"].mean()),
    "mean_top3_max_entailment": float(question_grounding_df["mean_top3_max_entailment"].mean()),
    "top3_supported_claim_rate": float(question_grounding_df["top3_supported_claim_rate"].mean()),
    "fully_top3_supported_answer_rate": float(question_grounding_df["fully_top3_supported"].mean()),
}])
grounding_summary = pd.concat([overall_grounding, grounding_summary], ignore_index=True)
grounding_summary.to_csv(TABLE_DIR / "locked_test_grounding_summary.csv", index=False)
display(grounding_summary.round(4))

## 28. Locked-test confidence calibration/discrimination — no refitting

In [ ]:
confidence_eval = confidence_df[
    ["id", "type", RANKING_SCORE, CALIBRATED_CONFIDENCE]
].merge(
    score_df[["id", "type", "task_score"]],
    on=["id", "type"],
    validate="one_to_one",
)

def ece_continuous(y, p, n_bins=10):
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i+1]
        mask = (p >= lo) & (p < hi if i < n_bins-1 else p <= hi)
        if mask.any():
            ece += mask.mean() * abs(y[mask].mean() - p[mask].mean())
    return float(ece)

conf_rows = []
for task in TASKS:
    g = confidence_eval[confidence_eval["type"] == task]
    y = g["task_score"].to_numpy(float)
    for method in [RANKING_SCORE, CALIBRATED_CONFIDENCE]:
        p = g[method].to_numpy(float)
        rho, pval = spearmanr(p, y) if np.std(p) > 1e-12 and np.std(y) > 1e-12 else (np.nan, np.nan)
        slope, intercept = (
            np.polyfit(p, y, 1) if np.std(p) > 1e-12 else (np.nan, np.nan)
        )
        conf_rows.append({
            "task": task,
            "method": method,
            "n": len(g),
            "mae": float(mean_absolute_error(y, p)),
            "rmse": float(np.sqrt(mean_squared_error(y, p))),
            "ece_10bin": ece_continuous(y, np.clip(p, 0, 1)),
            "calibration_bias": float(np.mean(p) - np.mean(y)),
            "calibration_intercept": float(intercept),
            "calibration_slope": float(slope),
            "spearman_rho": float(rho),
            "spearman_p": float(pval),
            "mean_confidence": float(np.mean(p)),
            "mean_observed_task_score": float(np.mean(y)),
        })

confidence_metrics_test = pd.DataFrame(conf_rows)
confidence_metrics_test.to_csv(TABLE_DIR / "locked_test_confidence_metrics_by_task.csv", index=False)
display(confidence_metrics_test.round(4))

yes = confidence_eval[confidence_eval["type"] == "yesno"]
yes_p = np.clip(yes[CALIBRATED_CONFIDENCE].to_numpy(float), 1e-8, 1-1e-8)
yes_y = yes["task_score"].to_numpy(float)
yesno_calibration_test = {
    "n": len(yes),
    "brier": float(np.mean((yes_p - yes_y) ** 2)),
    "log_loss": float(log_loss(yes_y, yes_p, labels=[0, 1])),
}
print("Yes/no binary calibration:", yesno_calibration_test)

## 29. Apply the already-frozen policy masks to test scores

In [ ]:
policy_masks = pd.read_csv(POLICY_MASK_CSV, dtype={"id": str})
policy_eval = policy_masks.merge(
    score_df[["id", "type", "task_score"]],
    on=["id", "type"],
    validate="many_to_one",
)

def evaluate_frozen_policy(frame, policy_name):
    p = frame[frame["policy"] == policy_name].copy()
    task_rows = []
    for task in TASKS:
        g = p[p["type"] == task]
        selected = g[g["selected_to_answer"].astype(bool)]
        task_rows.append({
            "policy": policy_name,
            "task": task,
            "n_total": int(len(g)),
            "n_answered": int(len(selected)),
            "n_abstained": int(len(g) - len(selected)),
            "coverage": float(len(selected) / len(g)),
            "full_task_score": float(g["task_score"].mean()),
            "selective_task_score": float(selected["task_score"].mean()),
        })
    per_task = pd.DataFrame(task_rows)
    full_macro = float(per_task["full_task_score"].mean())
    selective_macro = float(per_task["selective_task_score"].mean())
    return per_task, {
        "policy": policy_name,
        "n_total": int(len(p)),
        "n_answered": int(p["selected_to_answer"].astype(bool).sum()),
        "n_abstained": int((~p["selected_to_answer"].astype(bool)).sum()),
        "overall_question_coverage": float(p["selected_to_answer"].astype(bool).mean()),
        "overall_abstention_rate": float(1 - p["selected_to_answer"].astype(bool).mean()),
        "full_macro_task_score": full_macro,
        "selective_macro_task_score": selective_macro,
        "macro_task_score_gain": selective_macro - full_macro,
    }

pa_task, pa_summary = evaluate_frozen_policy(policy_eval, POLICY_A["name"])
pb_task, pb_summary = evaluate_frozen_policy(policy_eval, POLICY_B["name"])
policy_task_test = pd.concat([pa_task, pb_task], ignore_index=True)
policy_summary_test = pd.DataFrame([pa_summary, pb_summary])

policy_task_test.to_csv(TABLE_DIR / "locked_test_policy_task_results.csv", index=False)
policy_summary_test.to_csv(TABLE_DIR / "locked_test_policy_summary.csv", index=False)

display(policy_task_test.round(4))
display(policy_summary_test.round(6))

## 30. Paired bootstrap confidence intervals for frozen policy MacroTaskScore gain

In [ ]:
def bootstrap_frozen_policy_gain(frame, policy_name, n_boot=BOOTSTRAP_REPS, seed=SEED):
    p = frame[frame["policy"] == policy_name].copy()
    rng = np.random.default_rng(seed)
    gains = []

    for _ in range(n_boot):
        full_means, selected_means = [], []
        valid = True
        for task in TASKS:
            g = p[p["type"] == task].reset_index(drop=True)
            idx = rng.integers(0, len(g), size=len(g))
            b = g.iloc[idx]
            full_means.append(float(b["task_score"].mean()))
            sel = b[b["selected_to_answer"].astype(bool)]
            if len(sel) == 0:
                valid = False
                break
            selected_means.append(float(sel["task_score"].mean()))
        if valid:
            gains.append(float(np.mean(selected_means) - np.mean(full_means)))

    gains = np.asarray(gains, float)
    return {
        "policy": policy_name,
        "n_boot": int(len(gains)),
        "mean_gain": float(gains.mean()),
        "ci95_low": float(np.quantile(gains, 0.025)),
        "ci95_high": float(np.quantile(gains, 0.975)),
        "bootstrap_probability_gain_gt_0": float(np.mean(gains > 0)),
    }

policy_bootstrap_test = pd.DataFrame([
    bootstrap_frozen_policy_gain(policy_eval, POLICY_A["name"], seed=SEED),
    bootstrap_frozen_policy_gain(policy_eval, POLICY_B["name"], seed=SEED + 1),
])
policy_bootstrap_test.to_csv(TABLE_DIR / "locked_test_policy_macro_gain_bootstrap.csv", index=False)
display(policy_bootstrap_test)

## 31. Validation ↔ locked-test comparison

In [ ]:
validation_reference = {
    "retrieval_crossencoder_recall10": 0.6940663612120019,
    "full_macro_task_score": float(
        frozen_policy["policy_a_validation_summary"]["full_macro_task_score"]
    ),
    "policy_a_selective_macro_task_score": float(
        frozen_policy["policy_a_validation_summary"]["selective_macro_task_score"]
    ),
    "policy_a_gain": float(
        frozen_policy["policy_a_validation_summary"]["macro_task_score_gain"]
    ),
    "policy_b_selective_macro_task_score": float(
        frozen_policy["policy_b_validation_summary"]["selective_macro_task_score"]
    ),
    "policy_b_gain": float(
        frozen_policy["policy_b_validation_summary"]["macro_task_score_gain"]
    ),
}

test_reference = {
    "retrieval_crossencoder_recall10": float(retrieval_summary["recall@10"]),
    "full_macro_task_score": float(summary["macro_task_score"]),
    "policy_a_selective_macro_task_score": float(pa_summary["selective_macro_task_score"]),
    "policy_a_gain": float(pa_summary["macro_task_score_gain"]),
    "policy_b_selective_macro_task_score": float(pb_summary["selective_macro_task_score"]),
    "policy_b_gain": float(pb_summary["macro_task_score_gain"]),
}

comparison_rows = []
for metric in validation_reference:
    v = validation_reference[metric]
    t = test_reference[metric]
    comparison_rows.append({
        "metric": metric,
        "validation": v,
        "locked_test": t,
        "test_minus_validation": t - v,
    })
validation_test_comparison = pd.DataFrame(comparison_rows)
validation_test_comparison.to_csv(TABLE_DIR / "validation_vs_locked_test_comparison.csv", index=False)
display(validation_test_comparison.round(6))

## 32. Manuscript-ready figures

In [ ]:
# Figure 1 — task performance
task_test = pd.DataFrame({
    "task": ["Yes/no", "Factoid", "List", "Summary"],
    "score": [
        summary["yesno_accuracy"],
        summary["factoid_mrr"],
        summary["list_f1"],
        summary["summary_rougeL_f"],
    ],
})
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.bar(task_test["task"], task_test["score"])
ax.set_ylim(0, 1)
ax.set_ylabel("Locked-test task score")
ax.set_title("Frozen RAG system — one-time locked internal test")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig01_locked_test_task_scores.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure 2 — policy comparison
plot_policy = policy_summary_test.set_index("policy")[
    ["full_macro_task_score", "selective_macro_task_score"]
]
fig, ax = plt.subplots(figsize=(8, 4.8))
plot_policy.plot(kind="bar", ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("MacroTaskScore")
ax.set_xlabel("")
ax.set_title("Frozen selective-answering policies on locked test")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig02_locked_test_selective_policies.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure 3 — validation versus locked test
fig, ax = plt.subplots(figsize=(8, 4.8))
x = np.arange(len(validation_test_comparison))
w = 0.36
ax.bar(x - w/2, validation_test_comparison["validation"], width=w, label="Validation")
ax.bar(x + w/2, validation_test_comparison["locked_test"], width=w, label="Locked test")
ax.set_xticks(x)
ax.set_xticklabels(validation_test_comparison["metric"], rotation=25, ha="right")
ax.set_ylabel("Score / gain")
ax.set_title("Validation-to-locked-test transport of frozen pipeline")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "fig03_validation_vs_locked_test.png", dpi=300, bbox_inches="tight")
plt.show()

## 33. Final immutable Phase 5 report and completion marker

In [ ]:
phase5_report = {
    "dataset": "BioASQ14b",
    "locked_test_questions": 860,
    "evaluation_design": "one-time locked internal test after complete validation freeze",
    "stageA_manifest_sha256": sha256_file(STAGEA_MANIFEST),
    "locked_split_sha256": sha256_file(LOCKED_TEST_JSON),
    "frozen_pipeline": {
        "retrieval": "BM25 + MedCPT -> RRF -> MedCPT Cross-Encoder",
        "evidence_strategy": "document_top5",
        "generator": LLM_MODEL,
        "generation": "4-bit NF4; greedy; no sampling",
        "grounding": "Phase 4C v2; MedCPT fixed Top-3; rank-1 NLI primary",
        "confidence_ranking": RANKING_SCORE,
        "reported_confidence": CALIBRATED_CONFIDENCE,
    },
    "retrieval_locked_test": retrieval_summary,
    "answer_quality_locked_test": summary,
    "grounding_locked_test": grounding_summary.to_dict(orient="records"),
    "confidence_locked_test": confidence_metrics_test.to_dict(orient="records"),
    "yesno_binary_calibration_locked_test": yesno_calibration_test,
    "primary_policy": pa_summary,
    "secondary_policy": pb_summary,
    "policy_macro_gain_bootstrap": policy_bootstrap_test.to_dict(orient="records"),
    "validation_vs_locked_test": validation_test_comparison.to_dict(orient="records"),
    "post_hoc_test_tuning_performed": False,
    "locked_test_used_for_model_selection": False,
    "locked_test_used_for_coverage_selection": False,
    "locked_test_used_for_threshold_tuning": False,
    "interpretation_note": (
        "For factoid, list, and summary, task-specific calibrated confidence is an "
        "expected benchmark task score and not a probability of clinical truth."
    ),
    "completed_utc": datetime.now(timezone.utc).isoformat(),
}
save_json(phase5_report, FINAL_REPORT)

# Update Stage A manifest only to record that scoring has now happened.
manifest = load_json(STAGEA_MANIFEST)
manifest["gold_scoring_performed"] = True
manifest["gold_scoring_completed_utc"] = datetime.now(timezone.utc).isoformat()
save_json(manifest, STAGEA_MANIFEST)

marker = load_json(RUN_MARKER)
marker["status"] = "COMPLETE"
marker["completed_utc"] = datetime.now(timezone.utc).isoformat()
marker["final_report"] = str(FINAL_REPORT)
marker["final_report_sha256"] = sha256_file(FINAL_REPORT)
save_json(marker, RUN_MARKER)

print("PHASE 5 COMPLETE — LOCKED TEST EVALUATED ONCE.")
print("Final report:", FINAL_REPORT)
print("Final report SHA256:", sha256_file(FINAL_REPORT))

# STOP POINT 5 — Final outputs to send for review

After the one-time run, send these outputs/screenshots:

1. `locked_test_answer_summary.csv`
2. locked-test retrieval summary from Cell 26
3. `locked_test_grounding_summary.csv`
4. `locked_test_confidence_metrics_by_task.csv`
5. `locked_test_policy_summary.csv`
6. `locked_test_policy_macro_gain_bootstrap.csv`
7. `validation_vs_locked_test_comparison.csv`
8. all three Phase 5 figures
9. `phase5_final_locked_test_report.json`

**Do not modify the method after seeing these results.**  
Any later analysis must be explicitly labeled post-hoc/exploratory.